# 05 — LM-guided U-Net 학습 (Phase 3)

**목표**: Baseline (0.680 mIoU) 대비 **+3~5%p** 향상 검증.

**Baseline과 차이점 단 하나**: 입력 채널 3 → 4 (RGB + Landmark Heatmap).
나머지 (데이터, epoch, loss, augmentation) **모두 동일** → 공정 비교.

**예상 소요**: 
- 첫 실행: 랜드마크 캐시 빌드 (~15분) + 학습 (~60분) = **~75분**
- 재학습: 캐시 즉시 로드 → ~60분만

**Week 3 DoD**:
- ✅ best mIoU 보고
- ✅ Baseline vs LM-guided 비교표
- ✅ 예측 시각화 5장

## 1. 환경 셋업

In [ ]:
!nvidia-smi

In [ ]:
!pip install --quiet segmentation-models-pytorch albumentations datasets pyyaml mediapipe==0.10.21 tqdm

In [ ]:
import torch, segmentation_models_pytorch as smp, albumentations as A
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('smp:', smp.__version__)
print('albumentations:', A.__version__)
import mediapipe
print('mediapipe:', mediapipe.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/cv-final'
import sys
sys.path.insert(0, PROJECT_ROOT)

## 2. Landmark Heatmap 시각화 (sanity check)

한 장 샘플에 대해 RGB + Heatmap이 어떻게 생기는지 확인.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from project.segmentation.heatmap import landmark_heatmap

# 합성 랜드마크로 빠른 시각 검증
synthetic_lm = np.array([
    [128, 128],  # 코끝 위치
    [100, 100], [156, 100],  # 눈
    [128, 160],  # 입
    [128, 200],  # 턱
], dtype=np.int32)

hm = landmark_heatmap(synthetic_lm, size=256, sigma=3.0)
print('heatmap shape:', hm.shape, 'range:', hm.min(), '~', hm.max())

plt.figure(figsize=(6, 6))
plt.imshow(hm, cmap='hot')
plt.colorbar(label='intensity')
plt.title('Synthetic Landmark Heatmap (5 points, σ=3)')
plt.axis('off')
plt.show()

## 3. Dataset 검증 (4채널 입력 확인)

⚠️ **첫 실행 시 랜드마크 캐시 빌드 ~15분 소요**. 이후 즉시 로드.

In [ ]:
from project.segmentation import CelebAMaskHQDataset, get_train_transform

ds = CelebAMaskHQDataset(
    split='train',
    transform=get_train_transform(size=256, with_heatmap=True),
    subset_size=20,  # 빠른 검증용 (캐시 빌드는 전체 train에 대해 실행됨)
    cache_dir='/content/cv-final-cache',
    use_landmark=True,
    landmark_sigma=3.0,
)

img, mask = ds[0]
print('image shape:', img.shape, img.dtype)  # 기대: (4, 256, 256)
print('mask shape:', mask.shape, 'unique:', mask.unique().tolist())

assert img.shape[0] == 4, 'in_channels should be 4 (RGB + heatmap)'
print('✅ 4채널 입력 정상')

In [ ]:
# 4채널 시각화 — RGB + Heatmap
import torch

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(3):
    img, mask = ds[i]
    rgb = img[:3].permute(1, 2, 0).numpy()
    rgb = rgb * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    rgb = rgb.clip(0, 1)
    heatmap = img[3].numpy()

    axes[i].imshow(rgb)
    axes[i].imshow(heatmap, alpha=0.5, cmap='hot')
    axes[i].set_title(f'Sample {i} (RGB + Heatmap overlay)')
    axes[i].axis('off')

# 마지막 칸: heatmap만
_, _ = ds[0]
axes[3].imshow(ds[0][0][3].numpy(), cmap='hot')
axes[3].set_title('Sample 0 Heatmap only')
axes[3].axis('off')
plt.tight_layout()
plt.show()

## 4. 학습 실행 (LM-guided)

Baseline과 같은 조건 + use_landmark=True + Early Stopping.

In [ ]:
import yaml, os

CONFIG_PATH = f'{PROJECT_ROOT}/project/segmentation/config.yaml'
with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

# Phase 3 본 학습 설정 — Baseline과 동일하되 use_landmark만 true
cfg['dataset']['subset_size'] = None  # 전체 24K
cfg['train']['epochs'] = 20  # Early Stopping이 조기 종료
cfg['output']['checkpoint_dir'] = f'{PROJECT_ROOT}/project/segmentation/checkpoints'
cfg['dataset']['cache_dir'] = '/content/cv-final-cache'
cfg['model']['use_landmark'] = True   # ★ LM-guided ON
cfg['early_stopping']['enabled'] = True
cfg['early_stopping']['patience'] = 5

os.makedirs(cfg['output']['checkpoint_dir'], exist_ok=True)

TMP_CONFIG = '/tmp/config_lmguided.yaml'
with open(TMP_CONFIG, 'w') as f:
    yaml.safe_dump(cfg, f)

print('LM-guided config:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

In [ ]:
from project.segmentation.train import train

result = train(config_path=TMP_CONFIG)
print('\n=== LM-guided 학습 완료 ===')
print(f"Best mIoU:  {result['best_miou']:.4f} at epoch {result['best_epoch']}")
print(f"Best ckpt: {result['best_ckpt_path']}")

## 5. 학습 곡선 + Baseline 비교

In [ ]:
epochs = [h['epoch'] for h in result['history']]
losses = [h['loss'] for h in result['history']]
mious = [h['miou'] for h in result['history']]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epochs, losses, marker='o', color='#1E2761')
axes[0].set_title('LM-guided Training Loss'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].grid(alpha=0.3)
axes[1].plot(epochs, mious, marker='o', color='#F96167', label='LM-guided')
axes[1].axhline(y=0.680, color='gray', linestyle='--', label='Baseline best (0.680)')
axes[1].set_title('Validation mIoU'); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('mIoU')
axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout()
os.makedirs(f'{PROJECT_ROOT}/data/outputs', exist_ok=True)
plt.savefig(f'{PROJECT_ROOT}/data/outputs/lmunet_training_curve.png', dpi=150)
plt.show()

## 6. 비교 시각화 — Baseline vs LM-guided

In [ ]:
from project.segmentation.inference import load_checkpoint
from project.segmentation import get_val_transform

# Baseline 모델 로드 (이전에 학습한 unet_baseline_best.pt 또는 unet_v1.pt)
BASELINE_CKPT = f'{PROJECT_ROOT}/project/segmentation/checkpoints/unet_v1.pt'
LMGUIDED_CKPT = result['best_ckpt_path']

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Baseline (in_channels=3)
model_baseline = load_checkpoint(BASELINE_CKPT) if os.path.exists(BASELINE_CKPT) else None
if model_baseline is None:
    print(f'⚠️ Baseline checkpoint not found at {BASELINE_CKPT}. Skipping baseline comparison.')

# LM-guided (in_channels=4)
from project.segmentation.unet import build_unet
model_lm = build_unet(num_classes=6, in_channels=4, encoder_weights=None)
model_lm.load_state_dict(torch.load(LMGUIDED_CKPT, map_location=device))
model_lm.to(device).eval()

# 검증 데이터셋 (LM-guided용 → 4채널)
ds_viz_lm = CelebAMaskHQDataset(
    split='val',
    transform=get_val_transform(size=256, with_heatmap=True),
    subset_size=5,
    cache_dir='/content/cv-final-cache',
    use_landmark=True,
)
# 검증 데이터셋 (Baseline용 → 3채널)
ds_viz_base = CelebAMaskHQDataset(
    split='val',
    transform=get_val_transform(size=256, with_heatmap=False),
    subset_size=5,
    cache_dir='/content/cv-final-cache',
    use_landmark=False,
)

ncols = 4 if model_baseline is not None else 3
fig, axes = plt.subplots(5, ncols, figsize=(4*ncols, 18))
for i in range(5):
    img_lm, gt_mask = ds_viz_lm[i]
    rgb = img_lm[:3].permute(1, 2, 0).numpy()
    rgb = rgb * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    rgb = rgb.clip(0, 1)

    # LM-guided 예측
    with torch.no_grad():
        pred_lm = model_lm(img_lm.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()

    col = 0
    axes[i, col].imshow(rgb); axes[i, col].set_title('Input'); axes[i, col].axis('off'); col += 1
    axes[i, col].imshow(gt_mask.numpy(), cmap='tab10', vmin=0, vmax=5)
    axes[i, col].set_title('Ground Truth'); axes[i, col].axis('off'); col += 1

    if model_baseline is not None:
        img_base, _ = ds_viz_base[i]
        with torch.no_grad():
            pred_base = model_baseline(img_base.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()
        axes[i, col].imshow(pred_base, cmap='tab10', vmin=0, vmax=5)
        axes[i, col].set_title('Baseline'); axes[i, col].axis('off'); col += 1

    axes[i, col].imshow(pred_lm, cmap='tab10', vmin=0, vmax=5)
    axes[i, col].set_title('LM-guided ★'); axes[i, col].axis('off')

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/outputs/lmunet_predictions.png', dpi=150)
plt.show()

## 7. 최종 비교표 (Phase 4 시작 자료)

In [ ]:
BASELINE_MIOU = 0.680  # Phase 1 결과
LMGUIDED_MIOU = result['best_miou']
DELTA = LMGUIDED_MIOU - BASELINE_MIOU
DELTA_PCT = DELTA * 100

print('═' * 60)
print('   Baseline vs LM-guided 비교')
print('═' * 60)
print(f'  Baseline U-Net  (3채널) :  Val mIoU = {BASELINE_MIOU:.4f}')
print(f'  LM-guided U-Net (4채널) :  Val mIoU = {LMGUIDED_MIOU:.4f}')
print(f'  Δ                       :  {DELTA:+.4f}  ({DELTA_PCT:+.2f}%p)')
print('═' * 60)

if DELTA_PCT >= 3.0:
    print('  ✅ 큰 향상 (+3%p 이상) — LM-guided 채택, 추가 강화 진행 가능')
elif DELTA_PCT >= 1.0:
    print('  ⚠️  소폭 향상 (+1~3%p) — LM-guided 채택, GAN 후처리 강화 고려')
elif DELTA_PCT >= -1.0:
    print('  🟡 변화 미미 — 다른 강화 방향 검토 (Attention/Boundary)')
else:
    print('  ❌ 악화 — 구현 버그 검토 필요 (heatmap 정규화, 채널 순서)')

## 8. 완료 체크리스트 (Week 3 Phase 3)

- [ ] `nvidia-smi`에 T4 표시
- [ ] 모든 라이브러리 설치 + import 성공
- [ ] 합성 heatmap 시각화 (5점 → Gaussian 5개) 정상
- [ ] Dataset 4채널 입력 (`(4, 256, 256)`) 반환
- [ ] 랜드마크 캐시 빌드 (`landmarks.pkl`) 또는 캐시 hit
- [ ] LM-guided 학습 1 epoch 이상 완료 (loss 감소)
- [ ] Best checkpoint `unet_lmguided_best.pt` 저장
- [ ] Baseline vs LM-guided 비교표 출력
- [ ] 예측 시각화 5장 저장

**다음 단계 (Phase 4)**:
- 비교 결과 보고 → ΔmIoU 따라 다음 강화 결정
- 큰 향상이면 → Attention Gate / Boundary Loss 추가
- 미미하면 → GAN 후처리 (Refinement Network) 방향